# Example: Let's Build a Perceptron Classifier for Banknotes
Can a linear classifier tell a genuine banknote from a forged one? We'll use a banknote dataset to connect the Perceptron algorithm and the confusion matrix from [the lecture](CHEME-5800-L9a-Lecture-LinearClassificationAndPerceptron-Fall-2026.ipynb) to a numerical calculation.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * **Train a Perceptron classifier:** The Perceptron learns a linear decision boundary by updating its parameters each time a training example is misclassified or lies on the boundary. We'll train it on banknotes labeled genuine or forged, and see what happens when no hyperplane separates the two classes.
> * **Test the classifier on unseen data:** A classifier is only useful if it labels new examples correctly, so we hold back a testing set that the Perceptron never sees during training. We'll classify the testing banknotes and compare the mistakes on the testing set with the mistakes on the training set.
> * **Analyze the mistakes with a confusion matrix:** The confusion matrix separates the mistakes into false positives and false negatives. We'll compute it for the testing banknotes, calculate accuracy, precision, recall, and specificity, and interpret what each type of mistake means for banknote authentication.

In this example, we train the Perceptron on four numerical features measured from images of genuine and forged banknotes, classify banknotes it has not seen, and break down its mistakes with a confusion matrix. Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths -
include(joinpath(@__DIR__, "Include.jl")); # @__DIR__ locates this notebook's setup file

In addition to the Julia standard libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). See [the course package documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/), the [Julia documentation](https://docs.julialang.org/en/v1/), and the [DataFrames.jl documentation](https://dataframes.juliadata.org/stable/) for the functions and types used here.

### Data
We use the [banknote authentication dataset from the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/267/banknote+authentication), which holds 1,372 banknotes. Each banknote is described by four continuous features and labeled genuine or forged.

> __Dataset Description__
>
> * Images of genuine and forged banknote specimens were captured with an industrial camera used for print inspection. The final images are $400\times{400}$ pixel grayscale images with a resolution of about 660 dpi, and wavelet transform tools extracted the features from them.
> * __Features__: A wavelet transform breaks an image into patterns of different sizes and positions. The features are the `variance` (spread), `skewness` (asymmetry), and `curtosis` (tail heaviness) of the wavelet-transformed image, and the `entropy` (disorder) of the image; the dataset spells kurtosis as `curtosis`. The class is $y\in\{-1,1\}$, where $-1$ indicates genuine and $1$ indicates forged.

[The `MyBanknoteAuthenticationDataset(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyBanknoteAuthenticationDataset) loads this dataset and returns it [as a `DataFrame` instance](https://dataframes.juliadata.org/stable/). Let's save it in the `df_banknote::DataFrame` variable:

In [2]:
df_banknote = MyBanknoteAuthenticationDataset() # load the banknote dataset as a DataFrame

Row,variance,skewness,curtosis,entropy,class
,Float64,Float64,Float64,Float64,Int64
1,3.6216,8.6661,-2.8073,-0.44699,-1
2,4.5459,8.1674,-2.4586,-1.4621,-1
3,3.866,-2.6383,1.9242,0.10645,-1
4,3.4566,9.5228,-4.0112,-3.5944,-1
5,0.32924,-4.4552,4.5718,-0.9888,-1
6,4.3684,9.6718,-3.9606,-3.1625,-1
7,3.5912,3.0129,0.72888,0.56421,-1
8,2.0922,-6.81,8.4636,-0.60216,-1
9,3.2032,5.7588,-0.75345,-0.61251,-1


Now let's split the dataset into the data matrix $\mathbf{X}$ (the banknote features) and the label vector $\mathbf{y}$ (the banknote classes). The matrix $\mathbf{X}$ holds every column except `class`, and the vector $\mathbf{y}$ holds only the `class` column. We also count the banknotes in each class:

In [3]:
X = Matrix(df_banknote[:, Not(:class)]); # data matrix: select all the columns *except* class
y = Vector(df_banknote[:, :class]); # label vector: select the class column
println("genuine (y = -1): ", count(y .== -1), " banknotes, forged (y = 1): ", count(y .== 1), " banknotes")

genuine (y = -1): 762

 banknotes, forged (y = 1): 610 banknotes


The two classes are close in size: a little more than half of the banknotes are genuine.

Finally, let's partition the data into a `training` set and a `testing` set, so that we can measure how well the classifier labels banknotes it has not seen. We select the training banknotes at random with a seeded random number generator, so the split is the same every time we run the notebook.

We also append a column of ones to each data matrix, which gives the augmented features $\hat{\mathbf{x}}$ from the lecture, whose last entry multiplies the bias $b$. Each set is a `NamedTuple` with the fields `X` (the augmented data matrix) and `y` (the labels):

In [4]:
training, testing = let

    # initialize -
    s = 0.80; # fraction of data for training
    seed = 1234; # seed for the random number generator; change it to get a different split
    rng = MersenneTwister(seed); # random number generator used only for this split
    number_of_samples = size(X,1); # number of banknotes in the dataset
    number_of_training_samples = floor(Int, s*number_of_samples); # 80% of the data for training
    i = randperm(rng, number_of_samples); # random permutation of the row indices
    training_indices = i[1:number_of_training_samples]; # first 80% of the shuffled indices
    testing_indices = i[number_of_training_samples+1:end]; # last 20% of the shuffled indices

    # setup training -
    one_vector = ones(number_of_training_samples); # column of ones for the bias
    training = (X=[X[training_indices, :] one_vector], y=y[training_indices]);

    # setup testing -
    one_vector = ones(length(testing_indices)); # column of ones for the bias
    testing = (X=[X[testing_indices, :] one_vector], y=y[testing_indices]);

    # print the size of each set -
    println("training: ", length(training.y), " banknotes, testing: ", length(testing.y), " banknotes")

    training, testing; # return
end;

training: 1097 banknotes, testing: 275 banknotes


___

## Task 1: Train the Perceptron on the training banknotes
In this task, we train the Perceptron on the training banknotes with the online algorithm from the lecture, which looks at one banknote at a time and updates the parameters $\mathbf{\theta}$ whenever that banknote is a mistake. Here is the Perceptron learning algorithm pseudocode from the lecture:

__Initialize__: Given a dataset $\mathcal{D}$, the maximum number of passes through the data $T\geq{1}$, and a mistake threshold $M$ (we stop when a pass makes at most $M$ mistakes, e.g., $M=0$), initialize the parameter vector $\mathbf{\theta} = \left(w_{1},\dots,w_{m}, b\right)$ to zero or small random values and set the loop counter $t\gets{0}$.

While $\texttt{true}$ __do__:
1. Initialize the number of mistakes $\texttt{mistakes} \gets 0$.
2. For each training example $(\mathbf{x}, y) \in \mathcal{D}$, with augmented features $\hat{\mathbf{x}}$, __do__:
    - If $y\;\hat{\mathbf{x}}^{\top}\mathbf{\theta}\leq{0}$ __then__ the example is a __mistake__. Update $\mathbf{\theta} \gets \mathbf{\theta} + y\;\hat{\mathbf{x}}$ and increment $\texttt{mistakes} \gets \texttt{mistakes} + 1$.
3. After processing all examples, increment $t \gets t + 1$.
4. If $\texttt{mistakes} \leq {M}$ __or__ $t \geq T$ __then__ return $\mathbf{\theta}$. Otherwise, repeat from step 1.

__Training__: Our Perceptron implementation stores the problem data in [a `MyPerceptronClassificationModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyPerceptronClassificationModel). It holds the parameters $\mathbf{\theta}$ in its `β` field and the mistake threshold $M$ in its `mistakes` field. We build the model with [the `build(...)` factory method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build), starting from $\mathbf{\theta}=\mathbf{0}$ with $M=0$.

[The `learn(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.learn) runs the algorithm. It takes the augmented training data matrix, the training labels, and the model, and its `maxiter` argument sets the pass limit $T$. It returns the model with the learned parameters, which we save in the `model::MyPerceptronClassificationModel` variable:

In [5]:
model = let

    # initialize -
    X = training.X; # augmented training data matrix
    y = training.y; # training labels
    number_of_parameters = size(X,2); # p = m + 1: four weights and the bias
    T = 1000; # maximum number of passes through the training data
    M = 0; # mistake threshold: stop after a pass with at most M mistakes

    # build the model -
    model = build(MyPerceptronClassificationModel, (
        parameters = zeros(number_of_parameters), # initial parameters θ = 0: these will be updated
        mistakes = M # mistake threshold M
    ));

    # train -
    model = learn(X, y, model, maxiter = T, verbose = true); # verbose = true prints where the loop stopped

    model; # return
end;

Stopped after number of iterations: 1000

. We have number of errors: 12


__What do we see?__ The message counts each pass as an iteration: the Perceptron used all $T = 1000$ passes and never finished a pass without a mistake. These training banknotes are not linearly separable: we checked separately, by solving a small linear program, that no parameter vector puts every one of them on its correct side of the boundary. As the lecture showed, every pass then makes at least one mistake, so more passes cannot separate every training banknote.

The printed number of errors counts the mistakes made during the last pass, while $\mathbf{\theta}$ was still changing. To see how well the returned parameters fit the training data, let's count the training banknotes with $y\,\hat{\mathbf{x}}^{\top}\mathbf{\theta}\leq{0}$ using the learned parameters in `model.β`. We'll save the count in the `number_of_training_mistakes::Int64` variable:

In [6]:
number_of_training_mistakes = let

    # initialize -
    X = training.X; # augmented training data matrix
    y = training.y; # training labels
    θ = model.β; # learned parameters: w₁, w₂, w₃, w₄, then the bias b
    counter = 0; # number of mistakes

    # main loop: count the banknotes on the wrong side of the boundary, or on it -
    for i ∈ eachindex(y)
        x̂ = X[i,:]; # augmented features of banknote i
        if y[i]*dot(x̂, θ) ≤ 0
            counter += 1;
        end
    end

    # print the number of mistakes -
    println("training mistakes: ", counter, " of ", length(y), " banknotes")

    counter; # return
end;

training mistakes: 40 of 1097 banknotes


___

## Task 2: Classify the testing banknotes
In this task, we use the learned parameters to classify the testing banknotes, which the Perceptron never saw during training, and count the mistakes.

__Inference__: [The `classify(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.classify) takes the augmented testing data matrix and the trained model, and returns the estimated label $\hat{y}_{i} = \texttt{sign}(\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta})$ for each testing banknote. It uses [Julia's `sign(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.sign), which returns $0$ for a score of exactly zero, while the lecture's convention assigns the label $1$; we check for this case in Task 3. We store the actual labels in the `y_banknote::Vector{Int64}` variable and the estimated labels in the `ŷ_banknote::Vector{Float64}` variable:

In [7]:
ŷ_banknote, y_banknote = let

    # initialize -
    X = testing.X; # augmented testing data matrix
    y = testing.y; # actual labels

    # compute the estimated labels -
    ŷ = classify(X, model); # sign of the score for each testing banknote

    ŷ, y; # return
end;

How many mistakes did the classifier make on the testing set? We count the testing banknotes with $\hat{y}_{i}\neq{y}_{i}$, save the count in the `number_of_prediction_mistakes::Int64` variable, and print the mistake percentages for both sets:

In [8]:
number_of_prediction_mistakes = let

    # initialize -
    number_of_test_examples = length(ŷ_banknote); # how many testing banknotes?
    error_counter = 0; # number of mistakes

    # main loop: compare each estimated label with the actual label -
    for i ∈ 1:number_of_test_examples
        if (ŷ_banknote[i] != y_banknote[i])
            error_counter += 1;
        end
    end

    # print the mistake percentages -
    number_of_training_examples = length(training.y); # how many training banknotes?
    println("testing mistakes: ", error_counter, " of ", number_of_test_examples, " banknotes (", round(100*error_counter/number_of_test_examples, digits=1), "%)")
    println("training mistakes: ", number_of_training_mistakes, " of ", number_of_training_examples, " banknotes (", round(100*number_of_training_mistakes/number_of_training_examples, digits=1), "%)")

    error_counter; # return
end;

testing mistakes: 12 of 275 banknotes (4.4%)
training mistakes: 40 of 1097 banknotes (3.6%)


__What do we see?__ The Perceptron makes mistakes on a similar fraction of the testing and training banknotes, so it labels banknotes it has not seen about as well as the ones it was trained on. For these categorical predictions, we measure the error by counting wrong labels, rather than by computing residuals as we did for the continuous predictions of a regression model.

Because the data are not linearly separable, every pass updates the parameters at least once, so the returned parameters depend on where the loop stops.

___

## Task 3: Break down the mistakes with a confusion matrix
In this task, we compute the confusion matrix for the testing banknotes and the four performance metrics from the lecture. With the lecture's convention, the positive class is $y=1$, so a forged banknote is a positive and a genuine banknote is a negative.

> __Error analysis__: The total number of mistakes is only part of the story. A __false positive__ flags a genuine banknote as forged, which is a false alarm. A __false negative__ accepts a forged banknote as genuine, which lets a forgery through.

[The `confusion(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.confusion) takes the actual labels and the estimated labels, in that order. It returns the $2\times{2}$ confusion matrix laid out as in the lecture, with the actual classes in the rows and the predicted classes in the columns:
$$
\begin{equation*}
    \begin{bmatrix}
        TP & FN \\
        FP & TN
    \end{bmatrix}
\end{equation*}
$$
We save it in the `confusion_matrix::Matrix{Int64}` variable:

In [9]:
confusion_matrix = confusion(y_banknote, ŷ_banknote) # important: actual labels first, estimated labels second

2×2 Matrix{Int64}:
 121    9
   3  142

Next, let's read the four counts from the confusion matrix and compute accuracy, precision, recall, and specificity with the formulas from the lecture. We'll save the metrics in the `metrics::NamedTuple` variable:

In [10]:
metrics = let

    # initialize: read the counts from the confusion matrix -
    TP = confusion_matrix[1,1]; # forged banknotes labeled forged
    FN = confusion_matrix[1,2]; # forged banknotes labeled genuine (forgeries that get through)
    FP = confusion_matrix[2,1]; # genuine banknotes labeled forged (false alarms)
    TN = confusion_matrix[2,2]; # genuine banknotes labeled genuine

    # compute the metrics -
    accuracy = (TP + TN)/(TP + TN + FP + FN); # fraction of all predictions that are correct
    precision = TP/(TP + FP); # fraction of the forged predictions that are correct
    recall = TP/(TP + FN); # fraction of the forged banknotes that we catch
    specificity = TN/(TN + FP); # fraction of the genuine banknotes that we accept

    (accuracy = accuracy, precision = precision, recall = recall, specificity = specificity); # return
end;

Let's show the metrics in a table, next to the formulas from the lecture:

In [11]:
let

    # build the table -
    df = DataFrame(
        metric = ["accuracy", "precision", "recall", "specificity"], # metric names
        formula = ["(TP + TN)/(TP + TN + FP + FN)", "TP/(TP + FP)", "TP/(TP + FN)", "TN/(TN + FP)"], # formulas from the lecture
        value = round.([metrics.accuracy, metrics.precision, metrics.recall, metrics.specificity], digits=3) # computed values
    );

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = [:l, :l, :r] # left-align text and right-align numerical columns
    );
end

┌─────────────┬───────────────────────────────┬───────┐
│ metric      │ formula                       │ value │
├─────────────┼───────────────────────────────┼───────┤
│ accuracy    │ (TP + TN)/(TP + TN + FP + FN) │ 0.956 │
│ precision   │ TP/(TP + FP)                  │ 0.976 │
│ recall      │ TP/(TP + FN)                  │ 0.931 │
│ specificity │ TN/(TN + FP)                  │ 0.979 │
└─────────────┴───────────────────────────────┴───────┘


__What do we see?__ This is good performance for a simple linear classifier! The two classes are close in size, so accuracy is a fair summary here, unlike the rare-class case in the lecture.

Recall and specificity measure the mistakes within each class: one minus the recall is the fraction of forged banknotes that get through, and one minus the specificity is the fraction of genuine banknotes flagged as forged. Which kind of mistake matters more depends on what each one costs.

### Check: Labels and counts
Let's check that every estimated label is $1$ or $-1$, since the confusion matrix would skip a label of $0$. Then we check that the confusion matrix counts every testing banknote once, that its off-diagonal entries add up to the mistakes we counted in Task 2, and that the accuracy equals one minus the fraction of testing mistakes.

In [12]:
let

    # initialize -
    number_of_test_examples = length(y_banknote); # how many testing banknotes?
    FN = confusion_matrix[1,2]; # forged banknotes labeled genuine
    FP = confusion_matrix[2,1]; # genuine banknotes labeled forged

    # checks -
    @testset "Confusion matrix checks" begin
        @test all((ŷ_banknote .== 1) .| (ŷ_banknote .== -1)) # every estimated label is 1 or -1
        @test sum(confusion_matrix) == number_of_test_examples # every testing banknote is counted once
        @test FN + FP == number_of_prediction_mistakes # the off-diagonal entries are the mistakes
        @test metrics.accuracy ≈ 1 - number_of_prediction_mistakes/number_of_test_examples # accuracy = 1 - mistake fraction
    end
end;

Test Summary:           | Pass  

Total  Time
Confusion matrix checks |    4      4  0.4s


___

## Summary
In this example, we trained a Perceptron to tell genuine banknotes from forged ones, classified banknotes it had not seen, and broke down its mistakes with a confusion matrix.

> __Key Takeaways:__
>
> * **The banknotes are not linearly separable:** No hyperplane separates the training banknotes, so the Perceptron, trained from zero parameters with a zero-mistake threshold, used every pass without finishing one free of mistakes. Because the parameters keep changing during a pass, the mistake count printed for the last pass is not the training error of the parameters we get back.
> * **The classifier works about as well on new banknotes:** We classified the held-out testing banknotes and found a mistake rate similar to the training mistake rate. The similar mistake rates suggest that the learned boundary carries over to held-out banknotes from this dataset.
> * **The confusion matrix separates two kinds of mistakes:** We computed the confusion matrix for the testing banknotes, with forged banknotes as the positive class, and calculated accuracy, precision, recall, and specificity from its counts. We interpreted a false negative as a forgery that gets through and a false positive as a false alarm on a genuine banknote.

We can now change the seed of the split, the pass limit, or the mistake threshold and see how the parameters and the mistake counts respond.

___